# K1 — Cross-Family Replication on Llama-3.1-8B

**What this tests.** Prediction P2 of the CGF paper: the critical-rank window
where the proximity gap turns on is determined by the capacity ratio
$\rho = r_{LoRA} / d^*(\mathcal{D}_{ft})$, *not* by the model architecture.

We re-run the rank sweep E01–E04 (r ∈ {4, 16, 64, 128}) on
**Llama-3.1-8B-Instruct** with the same MedQA-USMLE corpus and
the same MMLU-57 evaluation. If P2 holds, we should see the proximity-gap
sign flip in the same window (between r=16 and r=64) on Llama as on Qwen.

**Setup**
- Free Kaggle T4 (16 GB) is enough at 4-bit.
- Add your Hugging Face token as a Kaggle secret named `HF_TOKEN`
  (Settings → Add-ons → Secrets) and accept the Llama-3.1 license at
  https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct.
- Estimated runtime: ~6 hours (1 baseline eval + 4 fine-tunes + 4 evals).


https://www.kaggle.com/code/akankshanarula/llama

In [ ]:
# Install the exact stack used in the paper. ~3 min on T4 / P100.
!pip install -q --upgrade \
    "transformers==4.46.3" "peft==0.13.2" "trl==0.11.4" "datasets==3.0.2" \
    "accelerate==1.0.1" "bitsandbytes==0.44.1" "sentencepiece" \
    "scipy" "scikit-learn" "matplotlib" "lm-eval==0.4.4" 2>&1 | tail -5
print("install done")


In [ ]:
import os, json, gc, time, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          BitsAndBytesConfig, TrainingArguments)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset, Dataset
from scipy.stats import friedmanchisquare, pearsonr
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "device", device)
print("gpu", torch.cuda.get_device_name(0) if device == "cuda" else "n/a")
print("vram", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")


In [ ]:
WORK = Path("/kaggle/working")
WORK.mkdir(exist_ok=True)
RESULTS_CSV = WORK / "results.csv"
PER_SUBJ_DIR = WORK / "per_subject"
PER_SUBJ_DIR.mkdir(exist_ok=True)

# Optional Hugging Face auth (needed for Llama-3.1-8B). Add HF_TOKEN as a Kaggle secret.
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print("HF token loaded from Kaggle secrets")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    print("no HF_TOKEN secret found; OK if all models are public")


In [ ]:
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

BNB = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

def load_base(model_id):
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=BNB, device_map="auto",
        torch_dtype=torch.bfloat16, trust_remote_code=True,
    )
    model.config.use_cache = False
    return tok, model

def attach_lora(model, r, target_modules=None):
    if target_modules is None:
        target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj"]
    cfg = LoraConfig(r=r, lora_alpha=r, lora_dropout=0.05,
                     bias="none", task_type="CAUSAL_LM",
                     target_modules=target_modules)
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, cfg)
    model.print_trainable_parameters()
    return model

def free():
    gc.collect(); torch.cuda.empty_cache()


In [ ]:
MMLU_SUBJECTS = [
    'abstract_algebra','anatomy','astronomy','business_ethics','clinical_knowledge',
    'college_biology','college_chemistry','college_computer_science','college_mathematics',
    'college_medicine','college_physics','computer_security','conceptual_physics',
    'econometrics','electrical_engineering','elementary_mathematics','formal_logic',
    'global_facts','high_school_biology','high_school_chemistry','high_school_computer_science',
    'high_school_european_history','high_school_geography','high_school_government_and_politics',
    'high_school_macroeconomics','high_school_mathematics','high_school_microeconomics',
    'high_school_physics','high_school_psychology','high_school_statistics',
    'high_school_us_history','high_school_world_history','human_aging','human_sexuality',
    'international_law','jurisprudence','logical_fallacies','machine_learning',
    'management','marketing','medical_genetics','miscellaneous','moral_disputes',
    'moral_scenarios','nutrition','philosophy','prehistory','professional_accounting',
    'professional_law','professional_medicine','professional_psychology','public_relations',
    'security_studies','sociology','us_foreign_policy','virology','world_religions'
]
MED_SUBJECTS = ['clinical_knowledge','medical_genetics','college_medicine','anatomy',
                'professional_medicine','virology','nutrition','human_aging','human_sexuality']

LETTERS = ["A","B","C","D"]

def fmt_q(row, with_answer=False):
    s = f"Question: {row['question']}\n"
    for i, c in enumerate(row['choices']):
        s += f"{LETTERS[i]}. {c}\n"
    s += "Answer:"
    if with_answer:
        s += f" {LETTERS[row['answer']]}\n\n"
    return s

def few_shot_prefix(dev_rows):
    return "".join(fmt_q(r, with_answer=True) for r in dev_rows)

@torch.no_grad()
def score_subject(model, tok, subject, n_shot=5, max_q=None):
    # 5-shot from MMLU 'dev' split, eval on 'test'.
    dev = load_dataset("cais/mmlu", subject, split="dev")
    test = load_dataset("cais/mmlu", subject, split="test")
    if max_q is not None:
        test = test.select(range(min(max_q, len(test))))
    shots = few_shot_prefix(list(dev.select(range(min(n_shot, len(dev))))))
    letter_ids = [tok(" " + L, add_special_tokens=False)["input_ids"][-1] for L in LETTERS]
    correct = 0
    for row in test:
        prompt = shots + fmt_q(row, with_answer=False)
        ids = tok(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
        out = model(**ids).logits[0, -1]
        pred = int(out[letter_ids].argmax().item())
        correct += int(pred == row['answer'])
    return correct / len(test)

def eval_all_mmlu(model, tok, tag, max_q_per=None):
    rows = []
    for s in MMLU_SUBJECTS:
        acc = score_subject(model, tok, s, max_q=max_q_per)
        rows.append({"subject": s, "acc": acc, "tag": tag})
        print(f"  {s:40s}  {acc:.3f}")
    df = pd.DataFrame(rows)
    df.to_csv(PER_SUBJ_DIR / f"{tag}.csv", index=False)
    return df


In [ ]:
def format_medqa(example):
    q = example["question"]
    opts = example["options"]
    if isinstance(opts, dict):
        opt_text = "\n".join(f"{k}. {v}" for k, v in opts.items())
    else:
        opt_text = "\n".join(f"{LETTERS[i]}. {c}" for i, c in enumerate(opts))
    ans = example.get("answer_idx") or example.get("answer", "")
    return {"text": f"Question: {q}\n{opt_text}\nAnswer: {ans}"}

def load_medqa(n=4096):
    ds = load_dataset("bigbio/med_qa", "med_qa_en_source", split="train",
                      trust_remote_code=True)
    ds = ds.select(range(min(n, len(ds))))
    ds = ds.map(format_medqa, remove_columns=ds.column_names)
    return ds

def run_finetune(model, tok, train_ds, steps=500, lr=2e-4, save_dir="/tmp/lora"):
    args = SFTConfig(
        output_dir=save_dir,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=lr, lr_scheduler_type="cosine",
        warmup_steps=10, max_steps=steps, logging_steps=20,
        save_strategy="no", report_to="none", bf16=True,
        max_seq_length=1024, dataset_text_field="text",
        seed=42,
    )
    trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=train_ds, args=args)
    trainer.train()
    return trainer


In [ ]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
RANKS = [4, 16, 64, 128]
STEPS = 500
SEED = 42
# Set this lower for a quick smoke test (e.g. 30) before the full run.
MAX_Q_PER_SUBJECT = None


In [ ]:
print(">>> Loading base for baseline MMLU eval")
set_seed(SEED)
tok, base = load_base(MODEL_ID)
print(">>> Baseline MMLU evaluation")
df_base = eval_all_mmlu(base, tok, tag="llama_base", max_q_per=MAX_Q_PER_SUBJECT)
del base; free()


In [ ]:
print(">>> Loading MedQA-USMLE")
train_ds = load_medqa(n=4096)
print(f"train size: {len(train_ds)}")

results = []
for r in RANKS:
    print(f"\n========== rank {r} ==========")
    set_seed(SEED)
    _, model = load_base(MODEL_ID)
    model = attach_lora(model, r=r)
    run_finetune(model, tok, train_ds, steps=STEPS, save_dir=f"/tmp/lora_r{r}")
    print(f">>> MMLU eval (rank={r})")
    df_ft = eval_all_mmlu(model, tok, tag=f"llama_r{r}_s500",
                          max_q_per=MAX_Q_PER_SUBJECT)
    # join with base
    j = df_ft.merge(df_base, on="subject", suffixes=("_ft","_base"))
    j["forgetting"] = j["acc_base"] - j["acc_ft"]
    j["is_med"] = j["subject"].isin(MED_SUBJECTS)
    f_med = j[j.is_med].forgetting.mean()
    f_non = j[~j.is_med].forgetting.mean()
    results.append({"rank": r, "f_mean": j.forgetting.mean(),
                    "f_med": f_med, "f_non": f_non,
                    "delta": f_med - f_non})
    j.to_csv(PER_SUBJ_DIR / f"llama_r{r}_diff.csv", index=False)
    del model; free()

res = pd.DataFrame(results)
res.to_csv(RESULTS_CSV, index=False)
print(res)


In [ ]:
# Friedman test across the 4 rank conditions (subject-stratified)
piv = pd.concat([
    pd.read_csv(PER_SUBJ_DIR / f"llama_r{r}_diff.csv")[["subject","forgetting"]]
       .rename(columns={"forgetting": f"r{r}"}) for r in RANKS
], axis=1)
piv = piv.loc[:, ~piv.columns.duplicated()]
piv = piv.dropna()
chi2, pval = friedmanchisquare(*[piv[f"r{r}"] for r in RANKS])
print(f"Friedman chi^2 = {chi2:.2f}, p = {pval:.2e}")

# Plot: mean forgetting + proximity gap
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(RANKS, res["f_mean"], "o-", lw=2, label="all (mean)")
axes[0].plot(RANKS, res["f_med"], "s--", lw=2, label="medical")
axes[0].plot(RANKS, res["f_non"], "v--", lw=2, label="non-medical")
axes[0].set_xscale("log"); axes[0].set_xlabel("LoRA rank")
axes[0].set_ylabel("Forgetting"); axes[0].legend(frameon=False)
axes[0].set_title("Llama-3.1-8B rank ablation")

axes[1].bar([str(r) for r in RANKS], res["delta"],
            color=["#1f77b4" if d < 0 else "#d62728" for d in res["delta"]])
axes[1].axhline(0, color="grey", lw=0.6)
axes[1].set_xlabel("LoRA rank"); axes[1].set_ylabel(r"$\Delta = f^{med} - f^{non}$")
axes[1].set_title("Proximity gap on Llama")
plt.tight_layout(); plt.savefig(WORK / "k1_summary.png", dpi=180); plt.show()


## What to look for

| Outcome | Interpretation |
|---|---|
| Δ flips sign between r=16 and r=64 | **P2 confirmed** — CGF universal across families |
| Δ flips at a different rank window | CGF holds but $\rho^*$ family-specific (interesting) |
| Δ never flips | **P2 falsified** — proximity gap is family-specific |

If you have the quota, repeat with `SEED ∈ {7, 13, 42}` and add error bars.
